# Model Boosting Experiments

Cuaderno para estudiar tecnicas de mejora sobre el pipeline base del TFG: aumento de datos, cambio de backbone y ajuste de configuraciones para modelos de `anomalib`.


## Objetivo del cuaderno

- documentar el bloque de mejoras descrito en la memoria;
- comparar alternativas de backbone y variantes de modelo;
- conservar una plantilla de HPO compatible con el estado actual del repo;
- evitar errores de ejecucion cuando falten checkpoints o resultados.


## Como leer este notebook

Este cuaderno traduce a codigo varias de las preguntas que aparecen en la memoria: si merece la pena cambiar de backbone, si el aumento de datos ayuda en un dataset pequeno y como se podria estructurar un ajuste de hiperparametros. Se prioriza la claridad del experimento sobre la automatizacion completa.


## Entorno y rutas


In [ ]:
from pathlib import Path
import importlib.util
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

DATA_DIR = ROOT / "data"
DOCS_DIR = ROOT / "docs"
ANOMALIB_DIR = ROOT / "anomalib"
RESULTS_DIR = ROOT / "results"
NOTES_DIR = ROOT / "notes"
NOTEBOOKS_DIR = ROOT / "notebooks"
NOTES_DIR.mkdir(exist_ok=True)

RAW_DATASET_DIR = DATA_DIR / "mandarins_pynq_raw"
CROPPED_DATASET_DIR = DATA_DIR / "mandarins_pynq_cropped"
AUGMENTED_DATASET_DIR = DATA_DIR / "mandarins_pynq_augmented"
INFERENCE_NORMAL_IMAGE = DATA_DIR / "inference_normal.png"
INFERENCE_ANOMALY_IMAGE = DATA_DIR / "inference_anomaly.png"

try:
    from anomalib.config import get_configurable_parameters as _anomalib_probe
    ANOMALIB_AVAILABLE = True
except Exception:
    ANOMALIB_AVAILABLE = False

TIMM_AVAILABLE = importlib.util.find_spec("timm") is not None
CV2_AVAILABLE = importlib.util.find_spec("cv2") is not None
KERAS_AVAILABLE = importlib.util.find_spec("keras") is not None

environment_summary = pd.DataFrame(
    [
        {"item": "repo_root", "value": str(ROOT)},
        {"item": "anomalib_available", "value": ANOMALIB_AVAILABLE},
        {"item": "timm_available", "value": TIMM_AVAILABLE},
        {"item": "cv2_available", "value": CV2_AVAILABLE},
        {"item": "keras_available", "value": KERAS_AVAILABLE},
        {"item": "raw_dataset_exists", "value": RAW_DATASET_DIR.exists()},
        {"item": "augmented_dataset_exists", "value": AUGMENTED_DATASET_DIR.exists()},
        {"item": "results_exists", "value": RESULTS_DIR.exists()},
    ]
)
environment_summary


La celda de entorno carga rutas comunes y fuerza el repo raiz en `sys.path`, lo que permite importar el modulo local `anomalib` aunque el notebook se ejecute desde `notebooks/` o desde una exportacion automatica con `nbconvert`.


## Resumen del dataset propio


In [ ]:
def dataset_count_table(dataset_dir: Path) -> pd.DataFrame:
    rows = []
    for label in ["normal", "abnormal"]:
        label_dir = dataset_dir / label
        count = len([path for path in sorted(label_dir.iterdir()) if path.is_file()]) if label_dir.exists() else 0
        rows.append({"dataset": dataset_dir.name, "label": label, "images": count})
    return pd.DataFrame(rows)


def summarize_mandarin_datasets() -> pd.DataFrame:
    frames = []
    for dataset_dir in [RAW_DATASET_DIR, CROPPED_DATASET_DIR, AUGMENTED_DATASET_DIR]:
        if dataset_dir.exists():
            frames.append(dataset_count_table(dataset_dir))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["dataset", "label", "images"])


def show_image_grid(image_paths, titles, figsize=(12, 4)) -> None:
    fig, axes = plt.subplots(1, len(image_paths), figsize=figsize)
    if len(image_paths) == 1:
        axes = [axes]
    for axis, image_path, title in zip(axes, image_paths, titles):
        axis.imshow(Image.open(image_path))
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()

boosting_dataset_summary = summarize_mandarin_datasets()
boosting_dataset_summary


## Resumen del bloque de mejoras del TFG


In [ ]:
boosting_topics = pd.DataFrame(
    [
        {"linea": "data_augmentation", "descripcion": "Generar nuevas muestras sinteticas sobre el dataset propio."},
        {"linea": "transfer_learning", "descripcion": "Cambiar el backbone para mejorar la representacion."},
        {"linea": "hyperparameter_tuning", "descripcion": "Ajustar configuraciones como score_type o pooling_kernel_size."},
        {"linea": "comparativa_mvtec", "descripcion": "Repetir el analisis sobre categorias de MVTec-AD."},
    ]
)
boosting_topics


Esta tabla resume el sentido del notebook: no es un simple cuaderno de entrenamiento, sino el bloque donde se materializan las tecnicas de mejora discutidas en el TFG.


## Imports opcionales para augmentation y modelos


In [ ]:
if CV2_AVAILABLE:
    import cv2
if TIMM_AVAILABLE:
    import timm
if KERAS_AVAILABLE:
    from keras.preprocessing.image import ImageDataGenerator
    from keras.utils import img_to_array, load_img
    from numpy import expand_dims

if ANOMALIB_AVAILABLE:
    from pytorch_lightning import Trainer
    from anomalib.config import get_configurable_parameters
    from anomalib.data.mvtec import MVTec
    from anomalib.models.cflow.lightning_model import Cflow
    from anomalib.models.dfm.lightning_model import Dfm
    from anomalib.models.fastflow.lightning_model import Fastflow
    from anomalib.models.patchcore.lightning_model import Patchcore
    
print({
    "cv2": CV2_AVAILABLE,
    "timm": TIMM_AVAILABLE,
    "keras": KERAS_AVAILABLE,
    "anomalib": ANOMALIB_AVAILABLE,
})


## Aumento de datos sobre mandarinas


En la memoria, el aumento de datos se plantea como una respuesta natural al tamano reducido del dataset propio. Aqui dejamos las funciones de trabajo para reproducir esa estrategia sin mezclarlas con las celdas de entrenamiento.


In [ ]:
if not (CV2_AVAILABLE and KERAS_AVAILABLE):
    print("Faltan dependencias opcionales para augmentation.")
else:
    def resize_images(source_dir: Path, target_dir: Path, image_size=(288, 288)) -> None:
        target_dir.mkdir(parents=True, exist_ok=True)
        for image_path in sorted(source_dir.iterdir()):
            if image_path.is_file():
                image = np.array(Image.open(image_path))
                resized = cv2.resize(image, image_size)
                Image.fromarray(resized).save(target_dir / image_path.name)

    def build_resized_dataset(source_dataset: Path, target_dataset: Path, image_size=(288, 288)) -> None:
        resize_images(source_dataset / "normal", target_dataset / "normal", image_size=image_size)
        resize_images(source_dataset / "abnormal", target_dataset / "abnormal", image_size=image_size)

    def augment_image(image_path: Path, destination_dir: Path, augmentations: int) -> None:
        image = img_to_array(load_img(image_path))
        batch_source = expand_dims(image, 0)
        augmenter = ImageDataGenerator(
            width_shift_range=0.2,
            height_shift_range=0.2,
            brightness_range=[0.4, 1.3],
            horizontal_flip=True,
            vertical_flip=True,
            fill_mode="nearest",
        )
        generated = 0
        for _ in augmenter.flow(batch_source, batch_size=1, save_prefix="augmented", save_to_dir=str(destination_dir), save_format="jpg"):
            generated += 1
            if generated >= augmentations:
                break

    print("Funciones de augmentation cargadas.")


## Exploracion de backbones


El cambio de backbone es una de las mejoras mas faciles de justificar en este proyecto, porque afecta directamente a la calidad de las representaciones sin obligar a redisenar todo el pipeline. La idea aqui es dejar visible que modelos estan al alcance y que capas exponen.


In [ ]:
if not TIMM_AVAILABLE:
    print("timm no esta disponible.")
else:
    candidate_backbones = ["resnet18", "resnet50", "efficientnet_b0", "efficientnet_l2"]
    backbone_overview = pd.DataFrame(
        [{"backbone": name, "available": name in timm.list_models()} for name in candidate_backbones]
    )
    backbone_overview


In [ ]:
if TIMM_AVAILABLE and "efficientnet_l2" in timm.list_models():
    efficientnet_l2 = timm.create_model("efficientnet_l2", pretrained=False)
    pd.DataFrame({"layer": [name for name, _ in efficientnet_l2.named_children()]})
else:
    print("efficientnet_l2 no esta disponible en timm.")


## Plantilla de entrenamiento sobre MVTec


La comparativa sobre `MVTec-AD` se deja como plantilla porque es la forma mas clara de conectar este notebook con las tablas de resultados del TFG. No se dispara entrenamiento automaticamente, pero quedan definidos los modelos ajustados y la logica necesaria para lanzarlos.


In [ ]:
if not ANOMALIB_AVAILABLE:
    print("anomalib no esta disponible.")
else:
    tuned_model_specs = [
        {
            "model_id": "dfm_fre",
            "model_class": "Dfm",
            "backbone": "resnet18",
            "layer_or_layers": "layer3",
            "notes": "DFM con score_type=frei y pooling_kernel_size=4",
        },
        {
            "model_id": "dfm_nll",
            "model_class": "Dfm",
            "backbone": "resnet18",
            "layer_or_layers": "layer3",
            "notes": "DFM con score_type=nll y pooling_kernel_size=2",
        },
        {
            "model_id": "patchcore_resnet18",
            "model_class": "Patchcore",
            "backbone": "resnet18",
            "layer_or_layers": "layer2, layer3",
            "notes": "Patchcore con coreset_sampling_ratio=0.3 y num_neighbors=18",
        },
        {
            "model_id": "cflow_resnet18",
            "model_class": "Cflow",
            "backbone": "resnet18",
            "layer_or_layers": "layer2, layer3, layer4",
            "notes": "CFlow sobre ResNet-18",
        },
        {
            "model_id": "fastflow_resnet18",
            "model_class": "Fastflow",
            "backbone": "resnet18",
            "layer_or_layers": "n/a",
            "notes": "FastFlow con weight_decay=0.0001",
        },
    ]

    pd.DataFrame(tuned_model_specs)


## Plantilla de HPO y WandB


Esta celda ya no intenta resolver todo el ajuste de hiperparametros, pero conserva un artefacto sencillo y legible que permite recordar como se estructuraria un barrido bayesiano sobre `DFM`.


In [ ]:
yaml_lines = [
    "program: tools/hpo/sweep.py",
    "method: bayes",
    "metric:",
    "  name: image_AUPR",
    "  goal: maximize",
    "parameters:",
    "  model.backbone:",
    "    values: [resnet18, efficientnet_l2]",
    "  model.score_type:",
    "    values: [fre, nll]",
    "  model.pooling_kernel_size:",
    "    values: [2, 4, 6]",
]

hpo_yaml_template = "\n".join(yaml_lines)
hpo_config_path = NOTES_DIR / "hpo_dfm_wandb.yaml"
hpo_config_path.write_text(hpo_yaml_template, encoding="utf-8")
pd.DataFrame([{"artifact": hpo_config_path.name, "path": str(hpo_config_path)}])


## Cierre

Este notebook queda centrado en la parte de mejora del TFG. Explica por que se exploran augmentation, backbones y HPO, y deja las piezas listas para repetir experimentos sin asumir que todo el entorno historico sigue intacto.
